# VISWIR: Visible and SWIR Image Fusion Demo 🚀

Welcome to the **VISWIR** (Visible and SWIR Weighted Image Reconstruction) image fusion demonstration notebook!

This interactive notebook guides you through setting up and running the **VISWIR** fusion pipeline. You will learn how to:
- **Install** the necessary packages.
- **Run the Fast Mode** pipeline on a single image pair to quickly see fusion results.
- **Run SQL Batch Mode** to process directories and save image metrics to a database.
- **Run Optuna Mode** to automatically optimize fusion parameters.

---

## Step 1: Installation & Repository Setup 🛠️

First, we will clone the public repository from GitHub and install all required python libraries (e.g., OpenCV, PyTorch, Scikit-Image, Optuna, SQLAlchemy, and imagecodecs for TIFF files).

In [ ]:
# Clone the repo and install dependencies
!git clone https://github.com/comsee-research/viswir.git
%cd viswir
!pip install -r requirements.txt
!pip install imagecodecs

## Step 2: Running the Fast Mode Pipeline 🏃‍♂️

**Fast Mode** (`--fast`) allows you to run the fusion on a single pair of visible and SWIR images without computing metrics or recording entries in the SQL database. This is great for rapid prototyping.

We will fuse the provided test images `data/VIS/clear.jpg` and `data/SWIR/clear.jpg` and output the result to `results/fused_clear.png`.

In [ ]:
# Run the fast pipeline
!python src/VISWIR_vQuasar.py --fast \
    --visible ./data/VIS/clear.jpg \
    --swir ./data/SWIR/clear.jpg \
    --out ./results/fused_clear.png

## Step 3: Visualizing the Results 🖼️

Let's compare the inputs (Visible and SWIR) alongside the fused VISWIR result.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Load images
vis = mpimg.imread("./data/VIS/clear.jpg")
swir = mpimg.imread("./data/SWIR/clear.jpg")
fused = mpimg.imread("./results/fused_clear.png")

# Plot
axes[0].imshow(vis)
axes[0].set_title("Visible Input")
axes[0].axis("off")

axes[1].imshow(swir, cmap="gray")
axes[1].set_title("SWIR Input")
axes[1].axis("off")

axes[2].imshow(fused)
axes[2].set_title("VISWIR Fused Result")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Step 4: SQL Mode Batch Processing 💾

**SQL Mode** processes all image pairs in the directories defined in `config/config_viswir.yaml` using the fusion parameters in `parameters.json`. It computes quality metrics (like SSIM, BRISQUE, NIQE, etc.) and logs all results to a SQLite database.

Let's write a minimal configuration file and execute the batch process.

In [ ]:
# Write batch config targeting the test dataset
config_content = """
visible_folder: "./data/VIS"
swir_folder: "./data/SWIR"
output_folder: "./results"
mode: "sql"
run_detection: false
save_output: true
"""

with open("config/config_viswir.yaml", "w") as f:
    f.write(config_content)

print("Config file created successfully!")

In [ ]:
# Run standard SQL mode
!python src/VISWIR_vQuasar.py

### Inspecting the Database

We can load the SQL results into a pandas DataFrame to inspect the calculated image quality metrics.

In [ ]:
import sqlite3
import pandas as pd

# Connect to results.db and load the table
conn = sqlite3.connect("/content/viswir/results/results.db")
df = pd.read_sql_query("SELECT * FROM fusion_results", conn)
conn.close()

# Display logs
df[['id', 'visible_img', 'alpha', 'beta', 'level', 'metrics_f']].head()

## Step 5: Optuna Hyperparameter Optimization ⚙️

**Optuna Mode** allows you to automatically optimize the parameters (such as SWIR weighting factor `alpha`/`facteur_swir`, `beta`, `level`, and `gamma`) to maximize the image quality metrics.

First, we will write the Optuna configuration and switch the main mode to `optuna` in the config file. Then we will run the runner.

In [ ]:
# Set Optuna configuration (10 trials for demonstration)
optuna_config = """
n_trials: 10
n_jobs: 1
sampler: "TPE"
pruner: "MedianPruner"
timeout: 600
storage: "sqlite:///results/optuna.db"
sample_size: 3
"""
with open("config/optuna_config.yaml", "w") as f:
    f.write(optuna_config)

# Switch configuration to Optuna mode
config_content_optuna = """
visible_folder: "./data/VIS"
swir_folder: "./data/SWIR"
output_folder: "./results"
mode: "optuna"
run_detection: false
save_output: false
"""
with open("config/config_viswir.yaml", "w") as f:
    f.write(config_content_optuna)

print("Optuna config set up successfully!")

In [ ]:
# Run Optuna optimization
!python src/VISWIR_vQuasar.py

### Visualizing the Optimization History 📊

Now, we can load the study from the SQLite database and plot the optimization history using Optuna's built-in visualization tools.

In [ ]:
import optuna

# Load and display study results
study = optuna.load_study(
    study_name="viswir_quality_optimization_study",
    storage="sqlite:///results/optuna.db"
)

print(f"Best trial value achieved: {study.best_value}")
print("Best parameters found:")
for k, v in study.best_params.items():
    print(f"  - {k}: {v}")

# Plot optimization history
optuna.visualization.plot_optimization_history(study)